In [ ]:
!nvidia-smi --query-gpu=name,compute_cap   
!git clone https://github.com/QuEST-Kit/QuEST.git

name, compute_cap
Tesla T4, 7.5
Cloning into 'QuEST'...
remote: Enumerating objects: 17409, done.
remote: Counting objects: 100% (1737/1737), done.
remote: Compressing objects: 100% (1023/1023), done.
remote: Total 17409 (delta 1253), reused 750 (delta 714), pack-reused 15672 (from 4)
Receiving objects: 100% (17409/17409), 19.35 MiB | 25.90 MiB/s, done.
Resolving deltas: 100% (11826/11826), done.


In [11]:
!find /content -name "qft_bench_naive.c" 2>/dev/null
%cd /content/QuEST
!ls

/content/qft_bench_naive.c
/content/QuEST
AUTHORS.txt  CMakeLists.txt  examples	  quest      tests
cmake	     docs	     LICENCE.txt  README.md  utils


In [ ]:
!cp /content/qft_bench_naive.c qft_bench_naive.c
!cmake -B build -DQUEST_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 \-DCMAKE_CUDA_FLAGS="-lineinfo" -DUSER_SOURCE_NAMES=qft_bench_naive.c -DUSER_OUTPUT_EXE_NAME=qft_naive
!cmake --build build -j4
!./build/qft_naive 24 5

In [19]:
!apt-get update -qq && apt-get install -y time

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  time
0 upgraded, 1 newly installed, 0 to remove and 71 not upgraded.
Need to get 45.2 kB of archives.
After this operation, 129 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 time amd64 1.9-0.2build1 [45.2 kB]
Fetched 45.2 kB in 0s (160 kB/s)
Selecting previously unselected package time.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../time_1.9-0.2build1_amd64.deb ...
Unpacking time (1.9-0.2build1) ...
Setting up time (1.9-0.2build1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [20]:
%%bash

EXE=/content/QuEST/build/qft_naive

echo "=== GPU memory before ==="
nvidia-smi --query-gpu=memory.used --format=csv,noheader

echo "=== System memory before ==="
free -h

echo "=== Running 28-qubit QFT ==="

/usr/bin/time -v "$EXE" 28 5 > /tmp/qft28_output.txt 2> /tmp/qft28_memory.txt &
PID=$!

GPU_PEAK=0

while kill -0 "$PID" 2>/dev/null; do
    USED=$(nvidia-smi \
        --query-gpu=memory.used \
        --format=csv,noheader,nounits | head -n 1)

    if [[ "$USED" =~ ^[0-9]+$ ]] && (( USED > GPU_PEAK )); then
        GPU_PEAK=$USED
    fi

    sleep 0.1
done

wait "$PID"
STATUS=$?

echo
echo "=== QFT benchmark output ==="
cat /tmp/qft28_output.txt

echo
echo "=== GPU memory peak ==="
echo "$GPU_PEAK MiB"

echo
echo "=== System memory peak (process RSS) ==="
grep "Maximum resident set size" /tmp/qft28_memory.txt

echo
echo "=== Exit status ==="
echo "$STATUS"

=== GPU memory before ===
0 MiB
=== System memory before ===
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       8.1Gi       2.3Mi       3.8Gi        11Gi
Swap:             0B          0B          0B
=== Running 28-qubit QFT ===

=== QFT benchmark output ===
QFT benchmark: 28 qubits, 5 reps, GPU backend, 4.00 GiB state vector
  rep 0: 8200.340 ms
  rep 1: 8193.259 ms
  rep 2: 8185.796 ms
  rep 3: 8178.002 ms
  rep 4: 8169.742 ms
  total prob    : 1.000000000000

  gates per QFT : 420
  best / mean   : 8169.742 / 8185.428 ms
  time per gate : 19451.768 us

=== GPU memory peak ===
4203 MiB

=== System memory peak (process RSS) ===
	Maximum resident set size (kbytes): 183492

=== Exit status ===
0


In [21]:
!nsys profile --trace=cuda,osrt --stats=true -o qft_24q ./build/qft_naive 24 5

QFT benchmark: 24 qubits, 5 reps, GPU backend, 0.25 GiB state vector
  rep 0:  385.017 ms
  rep 1:  385.014 ms
  rep 2:  385.023 ms
  rep 3:  384.936 ms
  rep 4:  384.999 ms
  total prob    : 1.000000000000

  gates per QFT : 312
  best / mean   : 384.936 / 384.998 ms
  time per gate : 1233.768 us
Generating '/tmp/nsys-report-0e9d.qdstrm'
[1/7] [========================100%] qft_24q.nsys-rep
[2/7] [========================100%] qft_24q.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)   Max (ns)    StdDev (ns)        Name     
 --------  ---------------  ---------  ------------  -------------  --------  -----------  ------------  --------------
     94.0    2,930,589,514         38  77,120,776.7  100,134,934.5   328,296  325,024,146  59,070,980.5  poll          
      5.9      183,378,732        508     360,981.8       14,712.0     1,259   18,291,169   1,266,552.5  ioctl         
      0.1        1,754,001      

In [22]:
!nsys profile --trace=cuda,osrt --stats=true -o qft_16q ./build/qft_naive 16 5

QFT benchmark: 16 qubits, 5 reps, GPU backend, 0.00 GiB state vector
  rep 0:    2.202 ms
  rep 1:    2.099 ms
  rep 2:    1.741 ms
  rep 3:    1.741 ms
  rep 4:    1.773 ms
  total prob    : 1.000000000000

  gates per QFT : 144
  best / mean   : 1.741 / 1.911 ms
  time per gate : 12.088 us
Generating '/tmp/nsys-report-c840.qdstrm'
[1/7] [========================100%] qft_16q.nsys-rep
[2/7] [========================100%] qft_16q.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -----------  --------  -----------  ------------  ----------------------
     73.6      524,177,776         14  37,441,269.7  8,242,973.5     1,548  323,061,398  84,931,538.6  poll                  
     25.7      182,880,086        501     365,030.1     17,571.0     1,030   21,739,329   1,319,031.4  ioctl                 
      0.3     

In [23]:
!nsys profile --trace=cuda,osrt --stats=true -o qft_28q ./build/qft_naive 28 5

QFT benchmark: 28 qubits, 5 reps, GPU backend, 4.00 GiB state vector
  rep 0: 8195.689 ms
  rep 1: 8195.787 ms
  rep 2: 8191.217 ms
  rep 3: 8184.746 ms
  rep 4: 8177.909 ms
  total prob    : 1.000000000000

  gates per QFT : 420
  best / mean   : 8177.909 / 8189.069 ms
  time per gate : 19471.211 us
Generating '/tmp/nsys-report-0beb.qdstrm'
[1/7] [========================100%] qft_28q.nsys-rep
[2/7] [========================100%] qft_28q.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls     Avg (ns)         Med (ns)        Min (ns)       Max (ns)     StdDev (ns)            Name         
 --------  ---------------  ---------  ---------------  ---------------  -------------  -------------  ------------  ----------------------
     50.8   49,881,857,800        507     98,386,307.3    100,136,061.0          1,516    291,605,394  16,260,963.6  poll                  
     48.9   48,002,508,914         12  4,000,209,076.2  4,000,082,828.0  4,000,019,536  4